In [2]:
from google.cloud import bigquery

client = bigquery.Client(project='musa5090s26-team5')

query = """
SELECT *
FROM `musa5090s26-team5.core.opa_properties`
LIMIT 5
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)
df.head()

,property_id,assessment_date,basements,beginning_point,book_and_page,building_code,building_code_description,category_code,category_code_description,census_tract,...,year_built,year_built_estimate,zip_code,zoning,pin,building_code_new,building_code_description_new,gdb_geomattr_data,objectid,geometry
0,871000021,2024-05-31T01:05:37Z,None,"204'4 1/4"" N BERKS",None,820,None,3,MIXED USE,157,...,2022,None,19122,RM1,1001681872,820,ROW MIXED-COM/RES-BLT AS RES,<NA>,579989,POINT(-75.137533 39.97998)
1,881000256,2024-12-13T12:40:55Z,None,None,53937673,None,None,14,APARTMENTS > 4 UNITS,369,...,None,None,19104,None,1001682367,None,None,<NA>,110807,POINT(-75.194611 39.954669)
2,783000004,2025-03-18T19:15:49Z,None,None,54197296,None,None,1,SINGLE FAMILY,102,...,None,None,19131,RM1,1001687937,None,None,<NA>,60520,POINT(-75.229957 39.969639)
3,881000277,2025-03-06T18:10:04Z,None,None,53937673,None,None,14,APARTMENTS > 4 UNITS,369,...,1970,None,19104,None,1001682368,829,APTS - HIGH RISE,<NA>,110806,POINT(-75.194651 39.954674)
4,874500512,2024-05-31T01:05:37Z,None,None,53793038,None,None,6,VACANT LAND,373,...,None,None,19145,I2,1001684632,None,None,<NA>,135484,POINT(-75.193549 39.905559)


In [3]:
query2 = """
SELECT
    COUNT(*) AS total_properties,
    COUNTIF(sale_price IS NULL) AS null_sale_price,
    COUNTIF(sale_price <= 1) AS dollar_sales,
    COUNTIF(sale_price > 1) AS valid_sales,
    MIN(sale_price) AS min_price,
    MAX(sale_price) AS max_price,
    AVG(sale_price) AS avg_price,
    APPROX_QUANTILES(sale_price, 4)[OFFSET(2)] AS median_price
FROM `musa5090s26-team5.core.opa_properties`
WHERE sale_price IS NOT NULL
"""

df2 = client.query(query2).to_dataframe(create_bqstorage_client=False)
df2

,total_properties,null_sale_price,dollar_sales,valid_sales,min_price,max_price,avg_price,median_price
0,581068,0,154862,426206,0.0,948729100.0,324237.0778,57900.0


In [5]:
query3 = """
SELECT
    sale_price,
    DATE(sale_date) AS sale_date,
    COUNT(*) AS property_count
FROM `musa5090s26-team5.core.opa_properties`
WHERE sale_price > 1
    AND sale_date IS NOT NULL
GROUP BY sale_price, DATE(sale_date)
HAVING COUNT(*) > 1
ORDER BY property_count DESC
LIMIT 20
"""

df3 = client.query(query3).to_dataframe(create_bqstorage_client=False)
df3

,sale_price,sale_date,property_count
0,4.0,1943-01-01,662
1,15925000.0,2005-12-21,317
2,9487291.0,1998-11-10,205
3,1683334.0,2011-05-31,187
4,26000000.0,2019-02-28,105
5,3.0,2016-10-14,93
6,8075000.0,2021-06-16,84
7,28500000.0,2013-04-26,77
8,3.0,2016-03-24,77
9,3.0,2016-06-23,65


In [6]:
query4 = """
WITH bundles AS (
    SELECT sale_price, DATE(sale_date) AS sale_date
    FROM `musa5090s26-team5.core.opa_properties`
    WHERE sale_price > 1 AND sale_date IS NOT NULL
    GROUP BY sale_price, DATE(sale_date)
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS clean_sales,
    MIN(DATE(sale_date)) AS earliest_sale,
    MAX(DATE(sale_date)) AS latest_sale,
    COUNTIF(DATE(sale_date) >= '2015-01-01') AS sales_since_2015,
    COUNTIF(DATE(sale_date) >= '2020-01-01') AS sales_since_2020
FROM `musa5090s26-team5.core.opa_properties` p
WHERE sale_price > 1
    AND sale_date IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM bundles b
        WHERE b.sale_price = p.sale_price
        AND b.sale_date = DATE(p.sale_date)
    )
"""

df4 = client.query(query4).to_dataframe(create_bqstorage_client=False)
df4

,clean_sales,earliest_sale,latest_sale,sales_since_2015,sales_since_2020
0,329795,1700-01-01,2026-03-11,157700,89347


In [7]:
query5 = """
SELECT
    COUNTIF(total_livable_area IS NULL) AS null_livable_area,
    COUNTIF(total_area IS NULL) AS null_total_area,
    COUNTIF(number_of_bedrooms IS NULL) AS null_bedrooms,
    COUNTIF(number_of_bathrooms IS NULL) AS null_bathrooms,
    COUNTIF(year_built IS NULL) AS null_year_built,
    COUNTIF(exterior_condition IS NULL) AS null_exterior_condition,
    COUNTIF(interior_condition IS NULL) AS null_interior_condition,
    COUNTIF(zip_code IS NULL) AS null_zip_code,
    COUNTIF(zoning IS NULL) AS null_zoning,
    COUNTIF(category_code IS NULL) AS null_category_code,
    COUNT(*) AS total
FROM `musa5090s26-team5.core.opa_properties`
"""

df5 = client.query(query5).to_dataframe(create_bqstorage_client=False)
df5.T

,0
null_livable_area,42888
null_total_area,521
null_bedrooms,81417
null_bathrooms,86579
null_year_built,42888
null_exterior_condition,43165
null_interior_condition,43187
null_zip_code,157
null_zoning,2280
null_category_code,0


# OPA Properties EDA - Feature Analysis

## Data Overview
- Total properties: 581,068
- All properties have a `sale_price` record

## Target Variable: `sale_price`
- Dollar-sales (≤$1): 154,862 (27%) — likely family transfers, exclude from model
- Bundle sales (same price + same date): identified and should be excluded
- Valid sales after filtering: ~329,795
- Median sale price: $57,900
- Mean sale price: $324,237 (skewed by high-value outliers)

## Data Freshness
- Sales data goes back to 1700 (unreliable historical records)
- Recommend using only sales since **2015** (157,700 records)
- Or only sales since **2020** (89,347 records) for stronger signal

## Recommended Features for Model

### High priority (low null rate)
- `total_area` — 521 nulls (<0.1%)
- `zip_code` — 157 nulls (<0.1%)
- `category_code` — 0 nulls (complete)
- `zoning` — 2,280 nulls (0.4%)

### Medium priority (7-8% null rate)
- `total_livable_area` — 42,888 nulls (7.4%)
- `year_built` — 42,888 nulls (7.4%)
- `exterior_condition` — 43,165 nulls (7.4%)
- `interior_condition` — 43,187 nulls (7.4%)

### Lower priority (high null rate)
- `number_of_bedrooms` — 81,417 nulls (14%)
- `number_of_bathrooms` — 86,579 nulls (14.8%)

## Data Cleaning Plan
1. Exclude dollar-sales (`sale_price <= 1`)
2. Exclude bundle sales (same `sale_price` + same `sale_date` for multiple properties)
3. Filter to sales since 2015 for training
4. Impute or drop rows with null values in key features
5. Focus on residential properties only (`category_code` = 1)